In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from pytorchvideo.data.clip_sampling import RandomClipSampler, UniformClipSampler

In [2]:
!pip install pytorchvideo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 43.3 MB/s eta 0:00:00:00:0100:01
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=e732f981e73e8dd8393d26a29c1f35da69abc8477323a57e7c4167ca81789bd6
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=304dbebdff7abed21ed0e6f78a890b9c142c53e25fef261d6dc5d057d6b25edf
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Crea

In [4]:
import torchvision

In [5]:
try:
    from torchvision.transforms import functional_tensor
except ImportError:
    import sys
    from torchvision.transforms import functional as F
    sys.modules["torchvision.transforms.functional_tensor"] = F

In [6]:
import torch
import numpy as np
import random

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42) 

In [8]:
class PackPathway(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, frames: torch.Tensor):
        # frames shape: [Channels, Time, Height, Width]
        
        # Fast Pathway: High frame rate (32 frames)
        fast_pathway = frames
        
        # Slow Pathway: Low frame rate (4 frames)
        # We pick 4 frames evenly spaced across the 32 frames
        slow_pathway = torch.index_select(
            frames,
            1, # Dimension 1 is Time
            torch.linspace(0, frames.shape[1] - 1, 4).long(),
        )
        
        return [slow_pathway, fast_pathway]

In [9]:
from pytorchvideo.transforms import ApplyTransformToKey, ShortSideScale, UniformTemporalSubsample
from torchvision.transforms import Compose, Lambda
from torchvision.transforms._transforms_video import CenterCropVideo, NormalizeVideo

# Paper specs: alpha=8, tau=16
# If we want 4 frames in Slow, we need 4 * 8 = 32 frames in Fast
num_frames_fast = 64

video_transform = ApplyTransformToKey(
    key="video",
    transform=Compose([
        # Pick 32 frames from the video
        UniformTemporalSubsample(num_frames_fast), 
        Lambda(lambda x: x/255.0),                 
        NormalizeVideo(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225]),
        ShortSideScale(size=256),
        CenterCropVideo(224),
        PackPathway()                              
    ]),
)

In [10]:
from pytorchvideo.data import Ucf101, make_clip_sampler

# Path to your data
train_path = "/kaggle/input/ucf101-subset/UCF101_subset/train"

train_dataset = Ucf101(
    data_path=train_path,
    clip_sampler=make_clip_sampler("random", 2.0), # Take a 2-second clip
    decode_audio=False,
    transform=video_transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8, 
    num_workers=2,
    pin_memory=True
)

# RES BLOCK


In [9]:
# res block

class res_block(nn.Module):
  def __init__(self,dim_in,dim_bottleneck ,dim_out,temp_size = 1, use_proj=False, spatial_stride = 1):
    super().__init__()
    t_pad = (temp_size - 1) // 2
    self.conv1 = nn.Conv3d(dim_in, dim_bottleneck, kernel_size=(temp_size, 1, 1),padding=(t_pad,0,0), bias=False,stride=1)
    self.batch_norm1 = nn.BatchNorm3d(dim_bottleneck)

    self.conv2 = nn.Conv3d(dim_bottleneck, dim_bottleneck, kernel_size=(1, 3, 3),stride=(1,spatial_stride,spatial_stride),padding=(0,1,1), bias=False)
    self.batch_norm2 = nn.BatchNorm3d(dim_bottleneck)

    self.conv3 = nn.Conv3d(dim_bottleneck, dim_out, kernel_size=(1, 1, 1), bias=False)
    self.batch_norm3 = nn.BatchNorm3d(dim_out)
    
    self.proj = None
    if use_proj:
      self.proj = nn.Conv3d(dim_in, dim_out, kernel_size=(1, 1, 1), stride=(1,spatial_stride,spatial_stride) ,bias=False)
      self.batch_norm4 = nn.BatchNorm3d(dim_out)

    self.relu = nn.ReLU()

  def forward(self, x):
    res = x
    x = self.conv1(x)
    x = self.batch_norm1(x)
    x = self.relu(x)

    x = self.conv2(x)
    x = self.batch_norm2(x)
    x = self.relu(x)

    x = self.conv3(x)
    x = self.batch_norm3(x)
    x = self.relu(x)
    if self.proj is not None:
      res = self.proj(res)
      res = self.batch_norm4(res)
    x = x+res
    x = self.relu(x)
    return x

In [10]:
# res_stage

class res_stage(nn.Module):
  def __init__(self,dim_in, dim_out, num_blocks,dim_bottleneck, temp_size=1,use_proj=False,spatial_stride = 1):
    super().__init__()
    self.res_stage = nn.ModuleList([
        res_block(dim_in=dim_in, dim_bottleneck=dim_bottleneck ,dim_out=dim_out, temp_size=temp_size ,use_proj = True, spatial_stride=spatial_stride) if i == 0
        else res_block(dim_in=dim_out,dim_bottleneck=dim_bottleneck, dim_out=dim_out, temp_size=temp_size, spatial_stride= 1)
        for i in range(num_blocks)
    ])

  def forward(self, x):
    for block in self.res_stage:
      x = block(x)
    return x

In [11]:
# res_stage

class res_stage(nn.Module):
  def __init__(self,dim_in, dim_out, num_blocks,dim_bottleneck, temp_size=1,use_proj=False,spatial_stride = 1):
    super().__init__()
    self.res_stage = nn.ModuleList([
        res_block(dim_in=dim_in, dim_bottleneck=dim_bottleneck ,dim_out=dim_out, temp_size=temp_size ,use_proj = True, spatial_stride=spatial_stride) if i == 0
        else res_block(dim_in=dim_out,dim_bottleneck=dim_bottleneck, dim_out=dim_out, temp_size=temp_size, spatial_stride= 1)
        for i in range(num_blocks)
    ])

  def forward(self, x):
    for block in self.res_stage:
      x = block(x)
    return x

In [12]:
class LateralConnection(nn.Module):
    def __init__(self, dim_in, alpha, beta_ratio=2):
        super().__init__()
        # The paper uses a 5x1x1 kernel
        # stride=alpha reduces the frame rate to match the Slow path
        # dim_out is usually beta_ratio * dim_in (making the connection "stronger")
        self.conv = nn.Conv3d(dim_in, dim_in * beta_ratio, 
                              kernel_size=(5, 1, 1), 
                              stride=(alpha, 1, 1), 
                              padding=(2, 0, 0), bias=False)
        self.bn = nn.BatchNorm3d(dim_in * beta_ratio)
        self.relu = nn.ReLU(inplace=F)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

In [13]:
#slowfast

class slowfast(nn.Module):
  def __init__(self, num_classes = 10):
    super().__init__()
    self.slowconv1 = nn.Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2),padding=(0, 3, 3), bias=False)
    self.slowpool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2),padding=(0, 1, 1))
    self.slowres2 = res_stage(dim_in=80, dim_bottleneck=64,dim_out=256, num_blocks=3)
    self.slowres3 = res_stage(dim_in=320,dim_bottleneck=128, dim_out=512, num_blocks=4, spatial_stride=2)
    self.slowres4 = res_stage(dim_in = 640,dim_bottleneck=256, dim_out=1024, num_blocks=6, temp_size= 3, spatial_stride=2)
    self.slowres5 = res_stage(dim_in = 1280,dim_bottleneck=512, dim_out=2048, num_blocks=3, temp_size= 3, spatial_stride=2)

    self.fastconv1 = nn.Conv3d(3, 8, kernel_size=(5, 7, 7), stride=(1, 2, 2),padding=(2, 3, 3), bias=False)
    self.fastpool1 = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2),padding=(0, 1, 1))
    self.fastres2 = res_stage(dim_in=8, dim_bottleneck=8,dim_out=32, num_blocks=3, temp_size= 3)
    self.fastres3 = res_stage(dim_in=32,dim_bottleneck=16, dim_out=64, num_blocks=4, temp_size= 3, spatial_stride=2)
    self.fastres4 = res_stage(dim_in = 64,dim_bottleneck=32, dim_out=128, num_blocks=6, temp_size= 3, spatial_stride=2)
    self.fastres5 = res_stage(dim_in = 128,dim_bottleneck=64, dim_out=256, num_blocks=3, temp_size= 3, spatial_stride=2)
    
    self.lateral_pool1 = LateralConnection(dim_in=8, beta_ratio=2, alpha=16)
    self.lateral_res2 = LateralConnection(dim_in=32, beta_ratio=2, alpha=16)
    self.lateral_res3 = LateralConnection(dim_in=64, beta_ratio=2, alpha=16)
    self.lateral_res4 = LateralConnection(dim_in=128, beta_ratio=2, alpha=16)

    self.avgpool = nn.AdaptiveAvgPool3d(1) # Reduces (T, H, W) to (1, 1, 1)
    self.dropout = nn.Dropout(p=0.5)
    self.fc = nn.Linear(2048 + 256, num_classes)

  def forward(self, x):
      
    slow_in, fast_in = x
      
    slow1 = self.slowconv1(slow_in)
    slow1 = self.slowpool1(slow1)
    fast1 = self.fastconv1(fast_in)
    fast1 = self.fastpool1(fast1)
    fuse1 = torch.cat([slow1, self.lateral_pool1(fast1)], dim=1)

    slow2 = self.slowres2(fuse1)
    fast2 = self.fastres2(fast1)
    fuse2 = torch.cat([slow2, self.lateral_res2(fast2)], dim=1)

    slow3 = self.slowres3(fuse2)
    fast3 = self.fastres3(fast2)
    fuse3 = torch.cat([slow3, self.lateral_res3(fast3)], dim=1)

    slow4 = self.slowres4(fuse3)
    fast4 = self.fastres4(fast3)
    fuse4 = torch.cat([slow4, self.lateral_res4(fast4)], dim=1)

    slow5 = self.slowres5(fuse4)
    fast5 = self.fastres5(fast4)
    
    slow_p = self.avgpool(slow5) 
    fast_p = self.avgpool(fast5) 

    
    slow_p = slow_p.view(slow_p.size(0), -1) # Shape: (N, 2048)
    fast_p = fast_p.view(fast_p.size(0), -1) # Shape: (N, 256)

    out = torch.cat([slow_p, fast_p], dim=1) # Shape: (N, 2304)

   
    out = self.dropout(out)
    out = self.fc(out)

    return out

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 10 
model = slowfast().to(device) 

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

criterion = nn.CrossEntropyLoss()

print(f"Model is on {device}. Ready to train.")

Model is on cuda. Ready to train.


In [ ]:
num_epochs= 30

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for i, batch in enumerate(train_loader):
        inputs = [x.to(device) for x in batch["video"]]
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        
        running_loss += loss.item()

        if i % 10 == 0:
            current_acc = 100 * correct_train / total_train
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i}], Loss: {loss.item():.4f}, Running Acc: {current_acc:.2f}%")

    avg_loss = running_loss / (i + 1)
    train_acc = 100 * correct_train / total_train
    print(f"--- Finished Epoch {epoch+1} | Avg Loss: {avg_loss:.4f} | Final Train Acc: {train_acc:.2f}% ---")

Epoch [1/30], Step [0], Loss: 3.0447, Running Acc: 12.50%
Epoch [1/30], Step [10], Loss: 2.4605, Running Acc: 10.23%
Epoch [1/30], Step [20], Loss: 2.1935, Running Acc: 12.50%
Epoch [1/30], Step [30], Loss: 2.1101, Running Acc: 14.11%
--- Finished Epoch 1 | Avg Loss: 2.5620 | Final Train Acc: 14.33% ---
Epoch [2/30], Step [0], Loss: 1.6503, Running Acc: 50.00%
Epoch [2/30], Step [10], Loss: 2.4959, Running Acc: 28.41%
Epoch [2/30], Step [20], Loss: 1.8879, Running Acc: 30.36%
Epoch [2/30], Step [30], Loss: 2.2207, Running Acc: 33.06%
--- Finished Epoch 2 | Avg Loss: 1.9882 | Final Train Acc: 32.33% ---
Epoch [3/30], Step [0], Loss: 1.9312, Running Acc: 50.00%
Epoch [3/30], Step [10], Loss: 1.4735, Running Acc: 40.91%
Epoch [3/30], Step [20], Loss: 1.5089, Running Acc: 35.71%
Epoch [3/30], Step [30], Loss: 2.0086, Running Acc: 33.87%
--- Finished Epoch 3 | Avg Loss: 1.9296 | Final Train Acc: 34.00% ---
Epoch [4/30], Step [0], Loss: 1.8889, Running Acc: 25.00%
Epoch [4/30], Step [10], Lo

In [ ]:
test_transform = video_transform 

# Create the test dataset
test_dataset = Ucf101(
    data_path="/kaggle/input/ucf101-subset/UCF101_subset/test",
    clip_sampler=make_clip_sampler("uniform", 2.0 ), 
    decode_audio=False,
    transform=test_transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=8,
    num_workers=2
)

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad(): 
    for i, batch in enumerate(test_loader):
        inputs = [x.to(device) for x in batch["video"]]
        labels = batch["label"].to(device)

        
        outputs = model(inputs)
        
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if i % 10 == 0:
            print(f"Batch {i} processed...")

final_accuracy = 100 * correct / total
print(f"Final Accuracy on Test Set: {final_accuracy:.2f}%")

In [ ]:
batch = next(iter(test_loader))
inputs = [x.to(device) for x in batch["video"]]
label = batch["label"][1].item()

output = model(inputs)
prediction = torch.argmax(output[0]).item()

print(f"Actual Label ID: {label}")
print(f"Model Prediction: {prediction}")